In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

df = pd.read_csv(f'{path}/Q3_data.csv')

In [ ]:
# Task 2: Write your code here:

print(df.head())


In [ ]:
# Task 3: Write your code here:
print(df.info())

In [ ]:
# Task 4: Write your code here:

print(df.describe())

In [ ]:
# Task 1: Write your code here:



In [ ]:
# Task 2: Write your code here:

df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'Target' in categorical_cols:
    categorical_cols.remove('Target')
if len(categorical_cols) > 0:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 4: Write your code here:

X = df.drop('Target', axis=1)
y = df['Target']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
# Task 5: Write your code here:

print("Target distribution:")
print(y.value_counts())
print(f"\nTarget imbalance ratio: {y.value_counts()[0] / y.value_counts()[1]:.2f}")
if y.value_counts()[0] / y.value_counts()[1] > 1.5:
    print("The target is IMBALANCED")
else:
    print("The target is BALANCED")

In [ ]:
# Task 1: Write your code here:

print(f"X shape: {X_scaled.shape}, y shape: {y.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:

skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skfold.split(X_scaled, y), 1):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)
    print(f"Fold {fold} - F1 Score: {f1:.4f}")

avg_f1 = np.mean(f1_scores)
print(f"Average F1 Score: {avg_f1:.4f}")

# Train final model for feature importance
final_model = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
final_model.fit(X_scaled, y)

In [ ]:
# Task 1: Write your code here:


feature_importance = pd.DataFrame({
    'feature': X_scaled.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(feature_importance['feature'][:20], feature_importance['importance'][:20])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 20 Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

golden_feature = feature_importance.iloc[0]['feature']
print(f"The Golden Feature is: {golden_feature}")


In [ ]:
# Task Bonus: Write your code here:


# Create new X with only golden feature
X_golden = X_scaled[[golden_feature]]

# Run KFold with single feature
golden_f1_scores = []

for fold, (train_idx, val_idx) in enumerate(skfold.split(X_golden, y), 1):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    f1 = f1_score(y_val, y_pred)
    golden_f1_scores.append(f1)
    print(f"Fold {fold} - Golden Feature F1: {f1:.4f}")

avg_golden_f1 = np.mean(golden_f1_scores)
print(f"\nAverage F1 Score with Golden Feature Only: {avg_golden_f1:.4f}")
print(f"Average F1 Score with All Features: {avg_f1:.4f}")
print(f"Difference: {avg_f1 - avg_golden_f1:.4f}")Claude is AI and can make mistakes. Please double-check responses.